# Prep

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import PyQt5
from pathlib import Path
from mne.channels import find_ch_adjacency
from functools import partial
import scipy.stats
import pickle
import seaborn as sns
from scipy import stats
import statistics
import os

#the following magic line will make all of your figures to be of the chosen size (in inches)
%matplotlib qt5

In [ ]:
path_to_preps = Path('D:/Studies/fake/exp_data/eeg_pretest/prep/filt')

In [ ]:
sbj_id = '049'

In [ ]:
raw = mne.io.read_raw_fif(path_to_preps / f'{sbj_id}.fif')

In [ ]:
raw.plot()

# Events and Epochs

In [ ]:
events, ids = mne.events_from_annotations(raw)

In [ ]:
ids

In [ ]:
dis_ids = {'t_avt': 5, 'f_avt': 6, 't_non': 7, 'f_non': 8}

In [ ]:
dis_events = mne.pick_events(events, include = [5, 6, 7, 8])

In [ ]:
mne.viz.plot_events(dis_events, event_id = dis_ids, sfreq = raw.info['sfreq'])

In [ ]:
epochs_dis = mne.Epochs(raw, dis_events, dis_ids, tmin = -0.2, tmax = 1, baseline = (-0.2, 0), reject_by_annotation=True)

In [ ]:
epochs_dis.plot()

# Evokeds

In [ ]:
evoked_dis = epochs_dis.average()
evoked_dis

In [ ]:
my_evokeds = {}
for condition in epochs_dis.event_id:
    my_evokeds[condition] =  epochs_dis[condition].average()
    
my_evokeds

mne.viz.plot_compare_evokeds(my_evokeds)

# Saving 1

In [ ]:
#epochs_dis.save(Path('D:/Studies/fake/exp_data/eeg_pretest/epoch')/ f'{sbj_id}.fif', overwrite=True)

In [ ]:
epochs_dis.save(Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo')/ f'{sbj_id}.fif', overwrite=True)

# Combine

In [ ]:
path_to_epochs = Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo')

In [ ]:
sbj_id = '001'
epochs_from_file = mne.read_epochs(path_to_epochs / f'{sbj_id}.fif', preload=False)

In [ ]:
changed_epochs = mne.epochs.combine_event_ids(epochs_from_file, ['t_avt', 't_non'], {'true': 35}, copy=True)

In [ ]:
ver_epochs = mne.epochs.combine_event_ids(changed_epochs, ['f_avt', 'f_non'], {'false': 46}, copy=True)

In [ ]:
ver_epochs

In [ ]:
my_evokeds = {}
for condition in ver_epochs.event_id:
    my_evokeds[condition] =  ver_epochs[condition].average()
    
my_evokeds

mne.viz.plot_compare_evokeds(my_evokeds)

In [ ]:
ver_epochs.plot()

In [ ]:
ver_epochs.save(Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/comb')/ f'{sbj_id}.fif', overwrite=True)

# ERP

In [ ]:
path_to_comb = Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/comb')
path_to_redo = Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/')
path_to_conf = Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/conf')

## All 4

In [ ]:
all_dis_2 = [file for file in path_to_redo.iterdir() if file.name.endswith('.fif')]
all_dis_2

In [ ]:
all_dis_epochs_2 = {file.name : None for file in all_dis_2}
for file in all_dis_2: 
    dis_epochs_2 = mne.read_epochs(file)
    all_dis_epochs_2[file.name] = dis_epochs_2

In [ ]:
all_dis_epochs_2

In [ ]:
all_dis_erps_2 = {name : {} for name in all_dis_epochs_2}
for name in all_dis_epochs_2:
    for event_id in all_dis_epochs_2[name].event_id:
        current_dis_epochs_2 = all_dis_epochs_2[name][event_id]
        dis_erp_2 = current_dis_epochs_2.copy().load_data().interpolate_bads().average()
        all_dis_erps_2[name][event_id] = dis_erp_2

In [ ]:
import pandas as pd

In [ ]:
erps_df = pd.DataFrame(all_dis_erps_2)

In [ ]:
erps_df.to_excel('D:/Studies/fake/exp_data/eeg_pretest/erps_df_2.xlsx', index=True)

In [ ]:
all_dis_erps_2

In [ ]:
grand_averages_2 = {event_id : mne.grand_average([evokeds[event_id] for evokeds in all_dis_erps_2.values()], drop_bads = False) for event_id in [[key for key in val.keys()] for ival, val in enumerate(all_dis_erps_2.values()) if ival == 0][0]}

In [ ]:
mne.viz.plot_compare_evokeds(
    {event_id : ga.copy() for event_id, ga in grand_averages_2.items()},
    colors=dict(t_avt=0, f_avt=1, t_non=0, f_non=1),
    linestyles=dict(t_avt='solid', f_avt='solid', t_non='dashed', f_non='dashed'),
    axes="topo",
    vlines = [0.0, 0.1, 0.3]
)

In [ ]:
mne.viz.plot_compare_evokeds({event_id : ga.copy().pick_channels(['Fz','FCz', 'Cz']) for event_id, ga in grand_averages_2.items()}, combine = 'mean',colors=dict(t_avt=0, f_avt=1, t_non=0, f_non=1),
    linestyles=dict(t_avt='solid', f_avt='solid', t_non='dashed', f_non='dashed'), vlines = [0.0, 0.1, 0.3])

In [ ]:
times = np.arange(0.05, 0.55, 0.05)
grand_averages_2['t_avt'].plot_topomap(times = times)

In [ ]:
grand_averages_2['f_avt'].plot_topomap(times = times)

In [ ]:
grand_averages_2['t_non'].plot_topomap(times = times)

In [ ]:
grand_averages_2['f_non'].plot_topomap(times = times)

# Difference

In [ ]:
differences_avt = [mne.combine_evoked([erp['f_avt'], erp['t_avt']], [1, -1]) for  erp in all_dis_erps_2.values()]

In [ ]:
differences_avt

In [ ]:
ga_differences_avt = mne.grand_average(differences_avt)

In [ ]:
ga_differences_avt

In [ ]:
mne.viz.plot_compare_evokeds(ga_differences_avt.copy().pick_channels(['Fz','FCz','Cz']),
                             combine = 'mean', 
                             title = 'f_avt - t_avt',
                             legend = None,
                            show_sensors = True)

In [ ]:
erps_df = pd.DataFrame(differences_avt)

In [ ]:
erps_df_2 = pd.DataFrame(differences_non)

In [ ]:
result = pd.concat([erps_df, erps_df_2], axis=1, join="inner")

In [ ]:
result.to_excel('D:/Studies/fake/exp_data/eeg_pretest/erps_diff.xlsx', index=True)

In [ ]:
differences_non = [mne.combine_evoked([erp['f_non'], erp['t_non']], [1, -1]) for erp in all_dis_erps_2.values()]

In [ ]:
ga_differences_non = mne.grand_average(differences_non)

In [ ]:
ga_differences_non

In [ ]:
times = np.arange(0.05, 0.75, 0.05)
ga_differences_avt.plot_topomap(times = times)

In [ ]:
ga_differences_non.plot_topomap(times = times)

In [ ]:
mne.viz.plot_compare_evokeds(ga_differences_non.copy().pick_channels(['Fz','FCz','Cz']),
                             combine = 'mean', 
                             title = 'f_non - t_non',
                             legend = None,
                            show_sensors = True)

In [ ]:
#f_avt - t_avt
times = np.arange(0.40, 0.65, 0.025)
ga_differences_avt['t_avt'].plot_topomap(times = times)

In [ ]:
#f_non - t_non
times = np.arange(0.05, 0.65, 0.05)
ga_differences_non['t_non'].plot_topomap(times = times)

In [ ]:
mne.viz.plot_compare_evokeds(
    {event_id : ga.copy().savgol_filter(12) for event_id, ga in grand_averages_2.items()},
    combine = 'mean',
    picks = ['Fz', 'FCz', 'Cz'],
    colors=dict(t_avt=0, f_avt=1, t_non=0, f_non=1),
    linestyles=dict(t_avt='solid', f_avt='solid', t_non='dashed', f_non='dashed'),
    show_sensors =  True
    #axes="topo",
    #vlines = [0.0, 0.2, 0.4]
)

# Analysis

## Diff

In [ ]:
with open('diff_avt_erps.pickle', 'wb') as container:
    pickle.dump(differences_avt, container, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
with open('diff_avt_erps.pickle', 'rb') as container:
    diff_avt_dict = pickle.load(container)

In [ ]:
with open('diff_non_erps.pickle', 'wb') as container:
    pickle.dump(differences_non, container, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
with open('diff_non_erps.pickle', 'rb') as container:
    diff_non_dict = pickle.load(container)

In [ ]:
loaded_dict

In [ ]:
diff_non_dict

In [ ]:
roi_comp_diff = []

vectors = []
for evoked in diff_avt_dict:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
avt_array = np.asarray(vectors)
roi_comp_diff.append(avt_array)

vectors = []
for evoked in diff_non_dict:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
non_array = np.asarray(vectors)
roi_comp_diff.append(non_array)

In [ ]:
def _custom_stat(a, b):
    res = scipy.stats.ttest_rel(a, b)
    return res.statistic
    
T_obs, clusters_2, cluster_p_values_2, H0 = mne.stats.permutation_cluster_test(
        roi_comp_diff,
        n_permutations=1024,
        stat_fun = _custom_stat,
        n_jobs=1,
        out_type="mask",
        threshold = 1.664,
        check_disjoint = True 
    )
clusters_2, cluster_p_values_2

In [ ]:
mne.viz.plot_compare_evokeds({'f_avt-t_avt' : ga_differences_avt, 'f_non-t_non' : ga_differences_non}, picks = ["Fz", "FCz", "Cz"], combine = 'mean')
times = f_avt_average.times
axes = plt.gca()
for i_c, c in enumerate(clusters_2):
    c = c[0]  # routine to get slice in 1-tuple
    if times[c.start] > 0 and (c.stop - c.start > 5):
        if cluster_p_values_2[i_c] <= 0.05:
            axes.axvspan(times[c.start], times[c.stop - 1], color="r", alpha=0.3),
        else:
            axes.axvspan(
                times[c.start],
                times[c.stop - 1],
                color=(0.1, 0.1, 0.1),
                alpha=0.1,
            ),
            print(times[c.start],times[c.stop - 1], cluster_p_values_2[i_c])

## ALL

In [ ]:
with open('all_dis_erps.pickle', 'wb') as container:
    pickle.dump(all_dis_erps_2, container, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
with open('all_dis_erps.pickle', 'rb') as container:
    loaded_dict = pickle.load(container)

In [ ]:
condition_list = ['t_avt', 'f_avt', 't_non', 'f_non']
condition_list

In [ ]:
t_avt_list = []
for i in condition_list:
    if i == 't_avt':
        tavt_evokeds = [loaded_dict[name][i].copy() for name in loaded_dict]
        t_avt_list = tavt_evokeds
        
f_avt_list = []
for i in condition_list:
    if i == 'f_avt':
        favt_evokeds = [loaded_dict[name][i].copy() for name in loaded_dict]
        f_avt_list = favt_evokeds
        
t_non_list = []
for i in condition_list:
    if i == 't_non':
        tnon_evokeds = [loaded_dict[name][i].copy() for name in loaded_dict]
        t_non_list = tnon_evokeds
        
f_non_list = []
for i in condition_list:
    if i == 'f_non':
        fnon_evokeds = [loaded_dict[name][i].copy() for name in loaded_dict]
        f_non_list = fnon_evokeds

In [ ]:
t_avt_list

In [ ]:
t_avt_average = mne.grand_average(t_avt_list)
f_avt_average = mne.grand_average(f_avt_list)
f_non_average = mne.grand_average(f_non_list)
t_non_average = mne.grand_average(t_non_list)

# PWC

pairwise comparison if more than 2 conditions

## FRN

In [ ]:
roi_comp_1 = []

vectors = []
for evoked in t_avt_list:
    avg_chan = evoked.copy().get_data().mean(axis = 0)
    vectors.append(avg_chan)
t_avt_array = np.asarray(vectors)
roi_comp_1.append(t_avt_array)

vectors = []
for evoked in f_avt_list:
    avg_chan = evoked.copy().get_data().mean(axis = 0)
    vectors.append(avg_chan)
f_avt_array = np.asarray(vectors)
roi_comp_1.append(f_avt_array)

In [ ]:
roi_comp_2 = []

vectors = []
for evoked in t_non_list:
    avg_chan = evoked.copy().get_data().mean(axis = 0)
    vectors.append(avg_chan)
t_non_array = np.asarray(vectors)
roi_comp_2.append(t_non_array)

vectors = []
for evoked in f_non_list:
    avg_chan = evoked.copy().get_data().mean(axis = 0)
    vectors.append(avg_chan)
f_non_array = np.asarray(vectors)
roi_comp_2.append(f_non_array)

In [ ]:
roi_comp_1 = []

vectors = []
for evoked in t_avt_list:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
t_avt_array = np.asarray(vectors)
roi_comp_1.append(t_avt_array)

vectors = []
for evoked in f_avt_list:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
f_avt_array = np.asarray(vectors)
roi_comp_1.append(f_avt_array)

In [ ]:
roi_comp_1

In [ ]:
roi_comp_2 = []
vectors = []
for evoked in t_non_list:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
t_non_array = np.asarray(vectors)
roi_comp_2.append(t_non_array)
    
vectors = []
for evoked in f_non_list:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
f_non_array = np.asarray(vectors)
roi_comp_2.append(f_non_array)

In [ ]:
roi_comp_3 = []
vectors = []
for evoked in t_avt_list:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
t_avt_array = np.asarray(vectors)
roi_comp_3.append(t_avt_array)
    
vectors = []
for evoked in t_non_list:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
t_non_array = np.asarray(vectors)
roi_comp_3.append(t_non_array)

In [ ]:
roi_comp_4 = []
vectors = []
for evoked in f_avt_list:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
f_avt_array = np.asarray(vectors)
roi_comp_4.append(f_avt_array)
    
vectors = []
for evoked in f_non_list:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
f_non_array = np.asarray(vectors)
roi_comp_4.append(f_non_array)

In [ ]:
roi_comp_5 = []
vectors = []
for evoked in t_avt_list:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
t_avt_array = np.asarray(vectors)
roi_comp_5.append(t_avt_array)
    
vectors = []
for evoked in f_non_list:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
f_non_array = np.asarray(vectors)
roi_comp_5.append(f_non_array)

In [ ]:
roi_comp_6 = []
vectors = []
for evoked in t_non_list:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
t_non_array = np.asarray(vectors)
roi_comp_6.append(t_non_array)
    
vectors = []
for evoked in f_avt_list:
    avg_chan = evoked.copy().pick_channels(["Fz", "FCz", "Cz"]).get_data().mean(axis = 0)
    vectors.append(avg_chan)
f_avt_array = np.asarray(vectors)
roi_comp_6.append(f_avt_array)

# Cluster

In [ ]:
roi_comp_1[0].shape

In [ ]:
def _custom_stat(a, b):
    res = scipy.stats.ttest_rel(a, b)
    return res.statistic
    
T_obs, clusters_2, cluster_p_values_2, H0 = mne.stats.permutation_cluster_test(
        [arr * 1e6 for arr in roi_comp_1],
        n_permutations=1024,
        stat_fun = _custom_stat,
        n_jobs=1,
        out_type="mask",
        threshold = 1.664,
        check_disjoint = True 
    )
clusters_2, cluster_p_values_2

In [ ]:
#[ar for ar in roi_comp_1]
times[clusters_2[6][0].start]

In [ ]:
mne.viz.plot_compare_evokeds({'true authoritative' : t_avt_average, 'false authoritative' : f_avt_average}, picks = ["Fz", "FCz", "Cz"], combine = 'mean')
times = f_avt_average.times
axes = plt.gca()
for i_c, c in enumerate(clusters_2):
    c = c[0]  # routine to get slice in 1-tuple
    if times[c.start] > 0 and (c.stop - c.start > 5):
        if cluster_p_values_2[i_c] <= 0.05:
            axes.axvspan(times[c.start], times[c.stop - 1], color="r", alpha=0.3),
            print(times[c.start])
        else:
            axes.axvspan(
                times[c.start],
                times[c.stop - 1],
                color=(0.1, 0.1, 0.1),
                alpha=0.1,
            ),
            print(times[c.start],times[c.stop - 1], cluster_p_values_2[i_c])

In [ ]:
def _custom_stat(a, b):
    res = scipy.stats.ttest_rel(a, b)
    return res.statistic
    
T_obs, clusters_2, cluster_p_values_2, H0 = mne.stats.permutation_cluster_test(
        [arr * 1e6 for arr in roi_comp_2],
        n_permutations=1024,
        stat_fun = _custom_stat,
        n_jobs=1,
        out_type="mask",
        threshold = 1.664,
        check_disjoint = True,
    )
clusters_2, cluster_p_values_2

In [ ]:
mne.viz.plot_compare_evokeds({'true non-authoritative' : t_non_average, 'false non-authoritative' : f_non_average}, picks = ["Fz", "FCz", "Cz"], combine = 'mean')
times = f_avt_average.times
axes = plt.gca()
for i_c, c in enumerate(clusters_2):
    c = c[0]  # routine to get slice in 1-tuple
    if times[c.start] > 0 and (c.stop - c.start > 5):
        if cluster_p_values_2[i_c] <= 0.05:
            axes.axvspan(times[c.start], times[c.stop - 1], color="r", alpha=0.3),
            print(times[c.start])
        else:
            axes.axvspan(
                times[c.start],
                times[c.stop - 1],
                color=(0.1, 0.1, 0.1),
                alpha=0.1,
            ),
            print(times[c.start],times[c.stop - 1], cluster_p_values_2[i_c])

In [ ]:
def _custom_stat(a, b):
    res = scipy.stats.ttest_rel(a, b)
    return res.statistic
    
T_obs, clusters_2, cluster_p_values_2, H0 = mne.stats.permutation_cluster_test(
        [arr * 1e6 for arr in roi_comp_3],
        n_permutations=1024,
        stat_fun = _custom_stat,
        n_jobs=1,
        out_type="mask",
        threshold = 1.664,
        check_disjoint = True,
    )
clusters_2, cluster_p_values_2

In [ ]:
mne.viz.plot_compare_evokeds({'true authoritative' : t_avt_average, 'true non-authoritative' : t_non_average}, picks = ["Fz", "FCz", "Cz"], combine = 'mean')
times = f_avt_average.times
axes = plt.gca()
for i_c, c in enumerate(clusters_2):
    c = c[0]  # routine to get slice in 1-tuple
    if times[c.start] > 0 and (c.stop - c.start > 5):
        if cluster_p_values_2[i_c] <= 0.05 / 6:
            axes.axvspan(times[c.start], times[c.stop - 1], color="r", alpha=0.3)
        else:
            axes.axvspan(
                times[c.start],
                times[c.stop - 1],
                color=(0.1, 0.1, 0.1),
                alpha=0.1,
            )

In [ ]:
def _custom_stat(a, b):
    res = scipy.stats.ttest_rel(a, b)
    return res.statistic
    
T_obs, clusters_2, cluster_p_values_2, H0 = mne.stats.permutation_cluster_test(
        [arr * 1e6 for arr in roi_comp_4],
        n_permutations=1024,
        stat_fun = _custom_stat,
        n_jobs=1,
        out_type="mask",
        threshold = 1.664,
        check_disjoint = True,
    )
clusters_2, cluster_p_values_2

In [ ]:
mne.viz.plot_compare_evokeds({'false authoritative' : f_avt_average, 'false non-authoritative' : f_non_average}, picks = ["Fz", "FCz", "Cz"], combine = 'mean')
times = f_avt_average.times
axes = plt.gca()
for i_c, c in enumerate(clusters_2):
    c = c[0]  # routine to get slice in 1-tuple
    if times[c.start] > 0 and (c.stop - c.start > 5):
        if cluster_p_values_2[i_c] <= 0.05 / 6:
            axes.axvspan(times[c.start], times[c.stop - 1], color="r", alpha=0.3)
        else:
            axes.axvspan(
                times[c.start],
                times[c.stop - 1],
                color=(0.1, 0.1, 0.1),
                alpha=0.1,
            )

In [ ]:
def _custom_stat(a, b):
    res = scipy.stats.ttest_rel(a, b)
    return res.statistic
    
T_obs, clusters_2, cluster_p_values_2, H0 = mne.stats.permutation_cluster_test(
        [arr * 1e6 for arr in roi_comp_5],
        n_permutations=1024,
        stat_fun = _custom_stat,
        n_jobs=1,
        out_type="mask",
        threshold = 1.664,
        check_disjoint = True,
    )
clusters_2, cluster_p_values_2

In [ ]:
mne.viz.plot_compare_evokeds({'true authoritative' : t_avt_average, 'false non-authoritative' : f_non_average}, picks = ["Fz", "FCz", "Cz"], combine = 'mean')
times = f_avt_average.times
axes = plt.gca()
for i_c, c in enumerate(clusters_2):
    c = c[0]  # routine to get slice in 1-tuple
    if times[c.start] > 0 and (c.stop - c.start > 5):
        if cluster_p_values_2[i_c] <= 0.05 / 6:
            axes.axvspan(times[c.start], times[c.stop - 1], color="r", alpha=0.3)
        else:
            axes.axvspan(
                times[c.start],
                times[c.stop - 1],
                color=(0.1, 0.1, 0.1),
                alpha=0.1,
            )

In [ ]:
def _custom_stat(a, b):
    res = scipy.stats.ttest_rel(a, b)
    return res.statistic
    
T_obs, clusters_2, cluster_p_values_2, H0 = mne.stats.permutation_cluster_test(
        [arr * 1e6 for arr in roi_comp_6],
        n_permutations=1024,
        stat_fun = _custom_stat,
        n_jobs=1,
        out_type="mask",
        threshold = 1.664,
        check_disjoint = True,
    )
clusters_2, cluster_p_values_2

In [ ]:
mne.viz.plot_compare_evokeds({'true non-authoritative' : t_non_average, 'false authoritative' : f_avt_average}, picks = ["Fz", "FCz", "Cz"], combine = 'mean')
times = f_avt_average.times
axes = plt.gca()
for i_c, c in enumerate(clusters_2):
    c = c[0]  # routine to get slice in 1-tuple
    if times[c.start] > 0 and (c.stop - c.start > 5):
        if cluster_p_values_2[i_c] <= 0.05 / 6:
            axes.axvspan(times[c.start], times[c.stop - 1], color="r", alpha=0.3)
        else:
            axes.axvspan(
                times[c.start],
                times[c.stop - 1],
                color=(0.1, 0.1, 0.1),
                alpha=0.1,
            )

# Permutations

In [ ]:
subj_wise_diffs_comp_1 = roi_comp_1[1] - roi_comp_1[0]

In [ ]:
subj_wise_diffs_comp_1.shape

In [ ]:
roi_1_arr = np.array(roi_comp_1)
roi_1_arr.shape

In [ ]:
T_obs, p_values, H0 = mne.stats.permutation_t_test(
    subj_wise_diffs_comp_1, 
    n_permutations=10000, 
    tail=0, 
    n_jobs=1)

In [ ]:
plt.plot(times, p_values)
plt.show()

# Strange analysis 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.stats
from mpl_toolkits.axes_grid1 import make_axes_locatable

import mne
from mne.channels import find_ch_adjacency
from mne.datasets import sample
from mne.stats import combine_adjacency, spatio_temporal_cluster_test
from mne.time_frequency import tfr_morlet
from mne.viz import plot_compare_evokeds

In [ ]:
new_comp_1 = []

mtxs = []
for evoked in t_avt_list:
    matrix = evoked.get_data()
    # print(matrix.shape)
    # print(matrix)
    mtxs.append(matrix)
# print([m.shape for m in mtxs])
t_avt_array = np.asarray(mtxs)
print(t_avt_array.shape)
new_comp_1.append(t_avt_array)

mtxs = []
for evoked in f_avt_list:
    matrix = evoked.get_data()
    mtxs.append(matrix)
f_avt_array = np.asarray(mtxs)
new_comp_1.append(f_avt_array)

In [ ]:
X = [np.transpose(x, (0, 2, 1)) for x in new_comp_1]
[x.shape for x in X]

In [ ]:
adjacency, ch_names = find_ch_adjacency(t_avt_average.info, ch_type="eeg")

print(type(adjacency))  # it's a sparse matrix!

#mne.viz.plot_ch_adjacency(t_avt_average.info, adjacency, ch_names)

In [ ]:
def _custom_stat(a, b):
    res = scipy.stats.ttest_rel(a, b)
    return res.statistic

# run the cluster based permutation analysis
cluster_stats = mne.stats.spatio_temporal_cluster_test(
    X,
    n_permutations=1000,
    stat_fun = _custom_stat,
    n_jobs=None,
    buffer_size=None,
    threshold = 1.664,
    adjacency=adjacency,
)
F_obs, clusters, p_values, _ = cluster_stats

In [ ]:
p_values

In [ ]:
# We subselect clusters that we consider significant at an arbitrarily
# picked alpha level: "p_accept".
# NOTE: remember the caveats with respect to "significant" clusters that
# we mentioned in the introduction of this tutorial!
p_accept = 0.4
good_cluster_inds = np.where(p_values < p_accept)[0]

# organize data for plotting
evokeds = {'true authoritative' : t_avt_average, 'false authoritative' : f_avt_average}

# loop over clusters
for i_clu, clu_idx in enumerate(good_cluster_inds):
    # unpack cluster information, get unique indices
    time_inds, space_inds = np.squeeze(clusters[clu_idx])
    ch_inds = np.unique(space_inds)
    time_inds = np.unique(time_inds)

    # get topography for F stat
    f_map = F_obs[time_inds, ...].mean(axis=0)

    # get signals at the sensors contributing to the cluster
    sig_times = t_avt_average.times[time_inds]

    # create spatial mask
    mask = np.zeros((f_map.shape[0], 1), dtype=bool)
    mask[ch_inds, :] = True

    # initialize figure
    fig, ax_topo = plt.subplots(1, 1, figsize=(10, 3), layout="constrained")

    # plot average test statistic and mark significant sensors
    f_evoked = mne.EvokedArray(f_map[:, np.newaxis], t_avt_average.info, tmin=0)
    f_evoked.plot_topomap(
        times=0,
        mask=mask,
        axes=ax_topo,
        cmap="Reds",
        vlim=(np.min, np.max),
        show=False,
        colorbar=False,
        mask_params=dict(markersize=10),
    )
    image = ax_topo.images[0]

    # remove the title that would otherwise say "0.000 s"
    ax_topo.set_title("")

    # create additional axes (for ERF and colorbar)
    divider = make_axes_locatable(ax_topo)

    # add axes for colorbar
    ax_colorbar = divider.append_axes("right", size="5%", pad=0.05)
    plt.colorbar(image, cax=ax_colorbar)
    ax_topo.set_xlabel(
        "Averaged F-map ({:0.3f} - {:0.3f} s)".format(*sig_times[[0, -1]])
    )

    # add new axis for time courses and plot time courses
    ax_signals = divider.append_axes("right", size="300%", pad=1.2)
    title = "Cluster #{0}, {1} sensor".format(i_clu + 1, len(ch_inds))
    if len(ch_inds) > 1:
        title += "s (mean)"
    plot_compare_evokeds(
        evokeds,
        title=title,
        picks=ch_inds,
        axes=ax_signals,
        show=False,
        split_legend=True,
        truncate_yaxis="auto",
    )

    # plot temporal cluster extent
    ymin, ymax = ax_signals.get_ylim()
    ax_signals.fill_betweenx(
        (ymin, ymax), sig_times[0], sig_times[-1], color="orange", alpha=0.3
    )

plt.show()

# Amplitudes

## FRN

In [ ]:
t_avt = 't_avt'
INTERVAL = [0.250, 0.350]

erps_tavt = [erps[t_avt].copy().pick_channels(["Fz", "FCz", "Cz"]) for _, erps in all_dis_erps_2.items()]
times = erps_tavt[0].times 
INTERVAL_idxs = [(np.abs(times - limit)).argmin() for limit in INTERVAL]

mean_amps_tavt = [erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].min() for erp in erps_tavt] # could use mas instead of mean

new_amps_tavt = []
lats_1 = []
for erp in erps_tavt:
    lat, mag = mne.preprocessing.peak_finder(erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].mean(axis = 0), extrema = -1)
    amp = np.min(mag)
    min_peak_idx = mag.argmin()
    new_amps_tavt.append(amp)
    lats_1.append(times[INTERVAL_idxs[0] + lat[min_peak_idx]])

In [ ]:
f_avt = 'f_avt'
INTERVAL = [0.250, 0.350]

erps_favt = [erps[f_avt].copy().pick_channels(["Fz", "FCz", "Cz"]) for _, erps in all_dis_erps_2.items()]
times = erps_favt[0].times 
INTERVAL_idxs = [(np.abs(times - limit)).argmin() for limit in INTERVAL]

mean_amps_favt = [erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].min() for erp in erps_favt] # could use mas instead of mean

new_amps_favt = []
lats_2 = []
for erp in erps_favt:
    lat, mag = mne.preprocessing.peak_finder(erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].mean(axis = 0), extrema = -1)
    amp = np.min(mag)
    new_amps_favt.append(amp)
    min_peak_idx = mag.argmin()
    lats_2.append(times[INTERVAL_idxs[0] + lat[min_peak_idx]])

In [ ]:
t_non = 't_non'
INTERVAL = [0.250, 0.350]

erps_tnon = [erps[t_non].copy().pick_channels(["Fz", "FCz", "Cz"]) for _, erps in all_dis_erps_2.items()]
times = erps_tnon[0].times 
INTERVAL_idxs = [(np.abs(times - limit)).argmin() for limit in INTERVAL]

mean_amps_tnon = [erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].min() for erp in erps_tnon] # could use mas instead of mean

new_amps_tnon = []
lats_3 = []
for erp in erps_tnon:
    lat, mag = mne.preprocessing.peak_finder(erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].mean(axis = 0), extrema = -1)
    amp = np.min(mag)
    new_amps_tnon.append(amp)
    min_peak_idx = mag.argmin()
    lats_3.append(times[INTERVAL_idxs[0] + lat[min_peak_idx]])

In [ ]:
f_non = 'f_non'
INTERVAL = [0.250, 0.350]

erps_fnon = [erps[f_non].copy().pick_channels(["Fz", "FCz", "Cz"]) for _, erps in all_dis_erps_2.items()]
times = erps_fnon[0].times 
INTERVAL_idxs = [(np.abs(times - limit)).argmin() for limit in INTERVAL]

mean_amps_fnon = [erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].min() for erp in erps_fnon] # could use mas instead of mean

new_amps_fnon = []
lats_4 = []
for erp in erps_fnon:
    lat, mag = mne.preprocessing.peak_finder(erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].mean(axis = 0), extrema = -1)
    amp = np.min(mag)
    new_amps_fnon.append(amp)
    min_peak_idx = mag.argmin()
    lats_4.append(times[INTERVAL_idxs[0] + lat[min_peak_idx]])

In [ ]:
statistics.mean(new_amps_fnon)

In [ ]:
len(mean_amps_tavt)

In [ ]:
statistics.mean(new_amps_favt)

In [ ]:
statistics.mean(lats_1)

In [ ]:
statistics.mean(lats_2)

In [ ]:
t_stat, p_value = stats.ttest_ind(mean_amps_tavt, mean_amps_favt)
p_value

In [ ]:
t_stat, p_value = stats.ttest_ind(new_amps_tavt, new_amps_favt)
p_value

## N400

In [ ]:
t_avt = 't_avt'
INTERVAL = [0.450, 0.520]

erps_tavt = [erps[t_avt].copy().pick_channels(["Fz", "FCz", "Cz"]) for _, erps in all_dis_erps_2.items()]
times = erps_tavt[0].times 
INTERVAL_idxs = [(np.abs(times - limit)).argmin() for limit in INTERVAL]

mean_amps_tavt = [erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].min() for erp in erps_tavt] # could use mas instead of mean

new_amps_tavt = []
lats_1 = []
for erp in erps_tavt:
    lat, mag = mne.preprocessing.peak_finder(erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].mean(axis = 0), extrema = -1)
    amp = np.min(mag)
    min_peak_idx = mag.argmin()
    new_amps_tavt.append(amp)
    lats_1.append(times[INTERVAL_idxs[0] + lat[min_peak_idx]])

In [ ]:
len(mean_amps_tavt)

In [ ]:
f_avt = 'f_avt'
INTERVAL = [0.450, 0.520]

erps_favt = [erps[f_avt].copy().pick_channels(["Fz", "FCz", "Cz"]) for _, erps in all_dis_erps_2.items()]
times = erps_favt[0].times 
INTERVAL_idxs = [(np.abs(times - limit)).argmin() for limit in INTERVAL]

mean_amps_favt = [erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].min() for erp in erps_favt] # could use mas instead of mean

new_amps_favt = []
lats_2 = []
for erp in erps_favt:
    lat, mag = mne.preprocessing.peak_finder(erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].mean(axis = 0), extrema = -1)
    amp = np.min(mag)
    new_amps_favt.append(amp)
    min_peak_idx = mag.argmin()
    lats_2.append(times[INTERVAL_idxs[0] + lat[min_peak_idx]])

In [ ]:
t_non = 't_non'
INTERVAL = [0.450, 0.520]

erps_tnon = [erps[t_non].copy().pick_channels(["Fz", "FCz", "Cz"]) for _, erps in all_dis_erps_2.items()]
times = erps_tnon[0].times 
INTERVAL_idxs = [(np.abs(times - limit)).argmin() for limit in INTERVAL]

mean_amps_tnon = [erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].min() for erp in erps_tnon] # could use mas instead of mean

new_amps_tnon = []
lats_3 = []
for erp in erps_tnon:
    lat, mag = mne.preprocessing.peak_finder(erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].mean(axis = 0), extrema = -1)
    amp = np.min(mag)
    new_amps_tnon.append(amp)
    min_peak_idx = mag.argmin()
    lats_3.append(times[INTERVAL_idxs[0] + lat[min_peak_idx]])

In [ ]:
f_non = 'f_non'
INTERVAL = [0.450, 0.520]

erps_fnon = [erps[f_non].copy().pick_channels(["Fz", "FCz", "Cz"]) for _, erps in all_dis_erps_2.items()]
times = erps_fnon[0].times 
INTERVAL_idxs = [(np.abs(times - limit)).argmin() for limit in INTERVAL]

mean_amps_fnon = [erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].min() for erp in erps_fnon] # could use mas instead of mean

new_amps_fnon = []
lats_4 = []
for erp in erps_fnon:
    lat, mag = mne.preprocessing.peak_finder(erp.data[:, INTERVAL_idxs[0] :INTERVAL_idxs[1]].mean(axis = 0), extrema = -1)
    amp = np.min(mag)
    new_amps_fnon.append(amp)
    min_peak_idx = mag.argmin()
    lats_4.append(times[INTERVAL_idxs[0] + lat[min_peak_idx]])

In [ ]:
lats_2

In [ ]:
statistics.mean(lats_1)

In [ ]:
statistics.mean(lats_2)

In [ ]:
t_stat, p_value = stats.ttest_ind(mean_amps_tavt, mean_amps_favt)
t_stat, p_value

In [ ]:
t_stat, p_value = stats.ttest_ind(new_amps_tavt, new_amps_favt)
t_stat, p_value

In [ ]:
t_stat, p_value = stats.ttest_ind(new_amps_tnon, new_amps_fnon)
t_stat, p_value

In [ ]:
from scipy.stats import wilcoxon

In [ ]:
dis = stats.shapiro(new_amps_favt)
dis

In [ ]:
res = wilcoxon(mean_amps_tavt, mean_amps_favt)
res

In [ ]:
res = wilcoxon(new_amps_tavt, new_amps_favt)
res

In [ ]:
res = wilcoxon(mean_amps_tnon, mean_amps_fnon)
res

In [ ]:
res = wilcoxon(new_amps_tnon, new_amps_fnon)
res

## To excel

In [ ]:
len(new_amps_fnon)

In [ ]:
import xlsxwriter
import pandas as pd

In [ ]:
means_frn = pd.DataFrame({'t_avt': mean_amps_tavt, 't_non': mean_amps_tnon, 'f_avt': mean_amps_favt, 'f_non': mean_amps_fnon})

# Export the DataFrame to an Excel file
means_frn.to_excel('D:/Studies/fake/exp_data/eeg_pretest/frn_means_new.xlsx', index=False)

mins_frn = pd.DataFrame({'t_avt': new_amps_tavt, 't_non': new_amps_tnon, 'f_avt': new_amps_favt, 'f_non': new_amps_fnon})

# Export the DataFrame to an Excel file
mins_frn.to_excel('D:/Studies/fake/exp_data/eeg_pretest/frn_mins_new.xlsx', index=False)

In [ ]:
means_n400 = pd.DataFrame({'t_avt': mean_amps_tavt, 't_non': mean_amps_tnon, 'f_avt': mean_amps_favt, 'f_non': mean_amps_fnon})

# Export the DataFrame to an Excel file
means_n400.to_excel('D:/Studies/fake/exp_data/eeg_pretest/n400_means_new.xlsx', index=False)

mins_n400 = pd.DataFrame({'t_avt': new_amps_tavt, 't_non': new_amps_tnon, 'f_avt': new_amps_favt, 'f_non': new_amps_fnon})

# Export the DataFrame to an Excel file
mins_n400.to_excel('D:/Studies/fake/exp_data/eeg_pretest/n400_mins_new.xlsx', index=False)

# Dinamics of t test

In [ ]:
# Simulate some ERP data for demonstration purposes
#n_channels = 32  # Number of electrodes
#n_times = 1000  # Number of time points
#n_subjects_group1 = 15  # Number of subjects in group 1
#n_subjects_group2 = 15  # Number of subjects in group 2



# Generate random data for two groups
data_tavt = np.asarray([erps['t_avt'].data for _, erps in all_dis_erps_2.items()])
data_favt = np.asarray([erps['f_avt'].data for _, erps in all_dis_erps_2.items()])
#times = erps_tavt[0].times 
#np.random.randn(n_subjects_group1, n_channels, n_times)
#data_group2 = #np.random.randn(n_subjects_group2, n_channels, n_times)

In [ ]:
mean_tavt = np.mean(data_tavt, axis=0)  # Average over subjects in group 1
mean_favt = np.mean(data_favt, axis=0)  # Average over subjects in group 2

In [ ]:
mean_tavt.shape

In [ ]:
from scipy.stats import ttest_ind

# Perform the t-test between two groups for each channel and time point
t_values, p_values = ttest_ind(data_tavt, data_favt, axis=0)  # axis=0 for subjects

In [ ]:
p_values.shape

In [ ]:
tavt = [erps['t_avt'] for _, erps in all_dis_erps_2.items()]
tavt

In [ ]:
tavt[0].ch_names

In [ ]:
info = mne.create_info(ch_names = tavt[0].ch_names, ch_types='eeg', sfreq = 500)

In [ ]:
info

In [ ]:
pseudo_evoked = mne.EvokedArray(data = t_values, info = info)

In [ ]:
pseudo_evoked_2 = mne.EvokedArray(data = p_values, info = info)

In [ ]:
pseudo_evoked

In [ ]:
montage = mne.channels.make_standard_montage("standard_1020")

In [ ]:
pseudo_evoked.set_montage(montage)
pseudo_evoked_2.set_montage(montage)

In [ ]:
pseudo_evoked.plot()

In [ ]:
pseudo_evoked_2.plot()

In [ ]:
import matplotlib.pyplot as plt

# Plot the t-values for all channels separately
for ichan in range(p_values.shape[0]):
    plt.plot(t_values[ichan, :])  # Plot for the first channel
    plt.xlabel("Time Points")
    plt.ylabel("T-Value")
    # plt.title(f"T-Test Results for Channel with index {ichan}")
plt.show()


In [ ]:
# Plot the p-values for a given channel over time
for ichan in range(p_values.shape[0]):
    plt.plot(p_values[ichan, :])  # Plot for the first channel
    plt.xlabel("Time Points")
    plt.ylabel("P-Value")
    plt.title("P-Value Results for Channel 1")
plt.show()

# Loreta

In [ ]:
from mne.datasets import sample
from mne.minimum_norm import apply_inverse, make_inverse_operator

## Covariance matrix

In [ ]:
all_cov = {name : {} for name in all_dis_epochs_2}
for name in all_dis_epochs_2:
    for event_id in all_dis_epochs_2[name].event_id:
        current_data = all_dis_epochs_2[name][event_id]
        noise_cov = mne.compute_covariance(
        current_data, tmax=0.0, method="auto", rank=None, verbose=True
)
        all_cov[name][event_id] = noise_cov

#fig_cov, fig_spectra = mne.viz.plot_cov(noise_cov, raw.info)

### New

In [ ]:
path_to_cov = Path('D:/Studies/fake/exp_data/eeg_pretest/loreta/cov')

In [ ]:
# Define the directory where you want to save covariance matrices
path_to_cov = Path('D:/Studies/fake/exp_data/eeg_pretest/loreta/cov')

# Ensure the save directory exists
path_to_cov.mkdir(parents=True, exist_ok=True)

for epoch_file in all_dis_2:
    epochs = mne.read_epochs(epoch_file, preload=True)
    
    # Compute covariance matrix
    cov = mne.compute_covariance(epochs, tmin=None, tmax=0, method='auto', rank=None, verbose=True)
    
    # Extract the base filename without the directory and extension
    base_filename = Path(epoch_file).stem  # This gets the filename without the directory and suffix

    # Save covariance matrix with custom naming
    save_path = path_to_cov / f'cov_{base_filename}.cov'  # Construct the full save path
    cov.save(save_path, overwrite = True)  # Save the covariance matrix


In [ ]:
all_cov

## EEG forward operator

In [ ]:
from mne.datasets import eegbci, fetch_fsaverage
import os.path as op
import nibabel

# Download fsaverage files
fs_dir = fetch_fsaverage(verbose=True)
subjects_dir = op.dirname(fs_dir)

# The files live in:
subject = "fsaverage"
trans = "fsaverage"  # MNE has a built-in fsaverage transformation
src = op.join(fs_dir, "bem", "fsaverage-ico-5-src.fif")
bem = op.join(fs_dir, "bem", "fsaverage-5120-5120-5120-bem-sol.fif")

In [ ]:
# Read and set the EEG electrode locations, which are already in fsaverage's
# space (MNI space) for standard_1020:
inv_list = {file.name : None for file in all_dis_2}
for file in all_dis_2:
    raw = mne.read_epochs(file, preload = True)
    raw.set_eeg_reference(projection=True)  # needed for inverse modeling
    inv_list[file.name] = raw

In [ ]:
#fwd_list = []
fwd_list = {}
for file in all_dis_2:
    fwd = mne.make_forward_solution(
        file, trans=trans, src=src, bem=bem, eeg=True, mindist=5.0, n_jobs=None
)
    fwd_list[file] = fwd

In [ ]:
new_fwd_list = {}
for key, value in fwd_list.items():
    new_key = os.path.basename(str(key))
    new_fwd_list[new_key] = value

# Replace the original fwd_list with the new one
fwd_list = new_fwd_list
new_fwd_list

In [ ]:
for eeg_file, fwd in new_fwd_list.items():
    mne.write_forward_solution(Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/fwd')/f'{eeg_file}_fwd.fif', fwd, overwrite = True)

In [ ]:
path_to_fwd = Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/fwd')

In [ ]:
fwd_names = [file for file in path_to_fwd.iterdir() if file.name.endswith('.fif')]
fwd_names

## Inverse operator

### Inverse with covariance

In [ ]:
path_to_fwd = Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/fwd')
path_to_cov = Path('D:/Studies/fake/exp_data/eeg_pretest/loreta/cov')

In [ ]:
cov_names = [file for file in path_to_cov.iterdir() if file.name.endswith('.cov')]
cov_names

In [ ]:
fwd_names = [file for file in path_to_fwd.iterdir() if file.name.endswith('.fif')]
fwd_names

In [ ]:
cov_list = {}
for cov_file in cov_names:
    cov = mne.read_cov(path_to_cov / f'{cov_file}')
    cov_list[cov_file.name] = cov

In [ ]:
cov_list

In [ ]:
inverse_operators_2 = []

for epoch, fwd_file, cov_file in zip(all_dis_2, fwd_names, cov_names):
    data = mne.read_epochs(epoch, preload = True)
    data.set_eeg_reference(projection=True)
    fwd = mne.read_forward_solution(fwd_file)
    cov = mne.read_cov(path_to_cov / f'{cov_file}')
    
    inverse_operator = mne.minimum_norm.make_inverse_operator(
        data.info, fwd, cov, loose=0.2, depth=0.8)
    
    #print(inverse_operator)
    inverse_operators_2.append(inverse_operator)

In [ ]:
inverse_operators_2

In [ ]:
inv = {}
for key, value in inverse_operators_2.items():
    new_key = os.path.basename(str(key))
    inv[new_key] = value

# Replace the original fwd_list with the new one
inverse_operators_2 = inv
inverse_operators_2

In [ ]:
for inv_number, inv in enumerate(inverse_operators_2):
    mne.minimum_norm.write_inverse_operator(Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/inv/inv_cov')/f'{str(inv_number + 1).zfill(3)}_inv.fif', inv, overwrite = True)

## Loreta

In [ ]:
path_to_loreta = Path('D:/Studies/fake/exp_data/eeg_pretest/loreta')

In [ ]:
inverse_operators = {}
for file in all_dis_erps_2:
    base_filename = file.replace('.fif', '')
    inv = mne.minimum_norm.read_inverse_operator(Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/inv/inv_cov')/f'{base_filename}_inv.fif')
    inverse_operators[file] = inv

In [ ]:
#fwd_saved_path = Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/fwd')
#inv_saved_path = Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/inv/inv_cov')

for eeg_file in all_dis_erps_2:
    #fwd = mne.read_forward_solution(Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/fwd')/f'{eeg_file}_fwd.fif')
    inv = mne.minimum_norm.read_inverse_operator(Path('D:/Studies/fake/exp_data/eeg_pretest/epoch/redo/inv/inv_cov')/f'{eeg_file}_inv.fif')
    #forward_operators[eeg_file] = fwd
    inverse_operators[eeg_file] = inv

#forward_operators = [mne.read_forward_solution(file) for file in fwd_saved_path.iterdir()]
#inverse_operators = [mne.minimum_norm.read_inverse_operator(file) for file in inv_saved_path.iterdir()]

In [ ]:
eeg_data = {file.name : None for file in all_dis_2}
for file in all_dis_2: 
    dis_epochs_2 = mne.read_epochs(file)
    #evoked = epochs.grand_average()
    eeg_data[file.name] = dis_epochs_2
    
evoked_data = {name : {} for name in eeg_data}
for name in eeg_data:
    for event_id in eeg_data[name].event_id:
        current_dis_epochs_2 = eeg_data[name][event_id]
        dis_erp_2 = current_dis_epochs_2.copy().load_data().interpolate_bads().average()
        evoked_data[name][event_id] = dis_erp_2    

In [ ]:
evoked_data

### SNR

In [ ]:
participants = list(evoked_data.keys())
conditions = ['t_avt', 'f_avt', 't_non', 'f_non']

In [ ]:
snr = {participant : {} for participant in eeg_data}
lambda_dict = {participant : {} for participant in eeg_data}
for participant in eeg_data:
    for event_id in eeg_data[participant].event_id:
        current_data = eeg_data[participant][event_id]
        snr_evoked = current_data.copy().average()
        mean_activation = snr_evoked.copy().crop(tmin=0, tmax=1).data.mean(axis=1)
        baseline_sd = snr_evoked.copy().crop(tmin=-0.2, tmax=0).data.std(axis=1)
        current_snr = np.mean(mean_activation / baseline_sd)
        snr[participant][event_id] = current_snr
        lambda2 = 1.0 / current_snr**2
        lambda_dict[participant][event_id] = lambda2

In [ ]:
snr

In [ ]:
lambda_dict

In [ ]:
# Directory to save the results
save_dir = path_to_loreta
#os.makedirs(save_dir, exist_ok=True)

participants = list(evoked_data.keys())
conditions = ['t_avt', 'f_avt', 't_non', 'f_non']

# Dictionary to store paths of saved results
saved_paths = {condition: [] for condition in conditions}

for participant in participants:
    for condition in conditions:
        # Get the evoked data
        evoked = evoked_data[participant][condition]
        evoked.set_eeg_reference(projection=True)
        
        # Get the corresponding inverse operator
        inverse_operator = inverse_operators[participant]
        
        # Get the lambda2 value for this participant and condition
        lambda2 = lambda_dict[participant][condition]
        
        # Compute the source estimate
        stc = mne.minimum_norm.apply_inverse(evoked, inverse_operator, lambda2=lambda2, method='sLORETA')
        
        # Save the source estimate to a file
        file_path = os.path.join(save_dir, f"{participant}_{condition}-stc.h5")
        stc.save(file_path, overwrite = True)
        
        # Store the path
        saved_paths[condition].append(file_path)

### No snr

In [ ]:
path_to_avg_loreta = Path('D:/Studies/fake/exp_data/eeg_pretest/loreta/avg')

In [ ]:
# Averaging the source estimates across participants for each condition
condition_averages = {}
save_dir = path_to_avg_loreta
os.makedirs(save_dir, exist_ok=True)

for condition in conditions:
    # Load the saved source estimates
    stcs = [mne.read_source_estimate(file_path) for file_path in saved_paths[condition]]
    
    # Average the source estimates manually
    mean_data = np.mean([stc.data for stc in stcs], axis=0)
    
    # Create a new SourceEstimate with the averaged data
    stc_avg = mne.SourceEstimate(mean_data, stcs[0].vertices, stcs[0].tmin, stcs[0].tstep, subject=stcs[0].subject)
    
    # Save the averaged source estimate
    avg_file_path = os.path.join(save_dir, f"{condition}_average-stc")
    stc_avg.save(avg_file_path, overwrite = True)
    
    # Store the averaged source estimate
    condition_averages[condition] = stc_avg

In [ ]:
# Directory where results are saved
save_dir = path_to_avg_loreta

# Load the averaged source estimates
condition_averages = {}

for condition in conditions:
    avg_file_path = os.path.join(save_dir, f"{condition}_average-stc")
    stc_avg = mne.read_source_estimate(avg_file_path)
    condition_averages[condition] = stc_avg

# Now you can use `condition_averages` for visualization or further analysis

In [ ]:
# Directory where results are saved
#path_to_loreta = Path("loreta_results")

# The subject to be used for visualization
subject = 'fsaverage'  # Change this to your subject's name if necessary

# Dictionary holding the conditions
conditions = ['t_avt', 'f_avt', 't_non', 'f_non'] #prints in the same order

# Create a directory for saving the plots
plot_save_dir = path_to_loreta / "plots"
plot_save_dir.mkdir(parents=True, exist_ok=True)

for condition in conditions:
    # Load the averaged source estimate
    stc = mne.read_source_estimate(path_to_avg_loreta / f'{condition}_average-stc')
    
    # Visualize the loaded LORETA results
    brain = mne.viz.plot_source_estimates(stc, subject=subject, surface='inflated', hemi = 'both', time_viewer=True, title = f'{condition}')
    # Add title to the plot
    brain.add_text(0.1, 0.9, f'Condition: {condition}', 'title', font_size=16)
    brain.show()
    
    # Save the plot with a condition-specific filename
    plot_path = plot_save_dir / f'{condition}_average.png'
    brain.save_image(plot_path)
    #brain.close()
    
    #2.84 -- f_avt
    #2.87 -- f_non
    #3.04 -- t_avt
    #2.93 -- t_non


### With covariance

In [ ]:
path_to_cov_loreta = Path('D:/Studies/fake/exp_data/eeg_pretest/loreta/loreta_cov')

In [ ]:
path_to_avg_cov_loreta = Path('D:/Studies/fake/exp_data/eeg_pretest/loreta/loreta_cov/avg')

In [ ]:
eeg_data = {file.name : None for file in all_dis_2}
for file in all_dis_2: 
    dis_epochs_2 = mne.read_epochs(file)
    #evoked = epochs.grand_average()
    eeg_data[file.name] = dis_epochs_2
    
evoked_data = {name : {} for name in eeg_data}
for name in eeg_data:
    for event_id in eeg_data[name].event_id:
        current_dis_epochs_2 = eeg_data[name][event_id]
        dis_erp_2 = current_dis_epochs_2.copy().load_data().interpolate_bads().average()
        evoked_data[name][event_id] = dis_erp_2    

In [ ]:
evoked_data

In [ ]:
# Directory to save the results
save_dir = path_to_cov_loreta
os.makedirs(save_dir, exist_ok=True)

participants = list(evoked_data.keys())
conditions = ['t_avt', 'f_avt', 't_non', 'f_non']

# Dictionary to store paths of saved results
saved_paths = {condition: [] for condition in conditions}

for participant in participants:
    for condition in conditions:
        # Get the evoked data
        evoked = evoked_data[participant][condition]
        evoked.set_eeg_reference(projection=True)
        
        # Get the corresponding inverse operator
        inverse_operator = inverse_operators[participant]
        
        # Compute the source estimate
        stc = mne.minimum_norm.apply_inverse(evoked, inverse_operator, lambda2=1.0 / 9.0, method='sLORETA')
        
        # Save the source estimate to a file
        file_path = os.path.join(save_dir, f"{participant}_{condition}-stc.h5")
        stc.save(file_path, overwrite =  True)
        
        # Store the path
        saved_paths[condition].append(file_path)

In [ ]:
participants = list(evoked_data.keys())
conditions = ['t_avt', 'f_avt', 't_non', 'f_non']
saved_paths = {condition: [] for condition in conditions}

In [ ]:
# Averaging the source estimates across participants for each condition
condition_averages = {}
save_dir = path_to_avg_cov_loreta
os.makedirs(save_dir, exist_ok=True)

for condition in conditions:
    # Load the saved source estimates
    stcs = [mne.read_source_estimate(file_path) for file_path in saved_paths[condition]]
    
    # Average the source estimates manually
    mean_data = np.mean([stc.data for stc in stcs], axis=0)
    
    # Create a new SourceEstimate with the averaged data
    stc_avg = mne.SourceEstimate(mean_data, stcs[0].vertices, stcs[0].tmin, stcs[0].tstep, subject=stcs[0].subject)
    
    # Save the averaged source estimate
    avg_file_path = os.path.join(save_dir, f"{condition}_average-stc")
    stc_avg.save(avg_file_path, overwrite = True)
    
    # Store the averaged source estimate
    condition_averages[condition] = stc_avg

In [ ]:
# Directory where results are saved
save_dir = path_to_avg_cov_loreta

# Load the averaged source estimates
condition_averages = {}

for condition in conditions:
    avg_file_path = os.path.join(save_dir, f"{condition}_average-stc")
    stc_avg = mne.read_source_estimate(avg_file_path)
    condition_averages[condition] = stc_avg

# Now you can use `condition_averages` for visualization or further analysis

In [ ]:
# Directory where results are saved
#path_to_loreta = Path("loreta_results")

# The subject to be used for visualization
subject = 'fsaverage'  # Change this to your subject's name if necessary

# Dictionary holding the conditions
conditions = ['t_avt', 'f_avt', 't_non', 'f_non'] #prints in the same order

# Create a directory for saving the plots
#plot_save_dir = path_to_cov_loreta / "plots"
#plot_save_dir.mkdir(parents=True, exist_ok=True)

for condition in conditions:
    # Load the averaged source estimate
    stc = mne.read_source_estimate(path_to_avg_cov_loreta / f'{condition}_average-stc')
    
    # Visualize the loaded LORETA results
    brain = mne.viz.plot_source_estimates(stc, subject=subject, surface='inflated', hemi = 'both', time_viewer=True, title = f'{condition}')
    # Add title to the plot
    brain.add_text(0.1, 0.9, f'Condition: {condition}', 'title', font_size=16)
    brain.show()
    
    # Save the plot with a condition-specific filename
    #plot_path = plot_save_dir / f'{condition}_average.png'
    #brain.save_image(plot_path)
    #brain.close()

In [ ]:
loreta_results = {}
file_names = []
conditions = []
for file_name, evoked_dict in evoked_data.items():
    for condition, evoked in evoked_dict.items():
        # Set the EEG reference to average using a projector
        evoked.set_eeg_reference(projection=True)
        stc = mne.minimum_norm.apply_inverse(
            evoked, inverse_operators[file_name], lambda2=1.0 / 9, method='sLORETA'
        )
        loreta_results[f'{file_name}_{condition}'] = stc

        file_names.append(file_name)
        conditions.append(condition)

        #stc.save(path_to_loreta / f'{file_name}_{condition}_loreta.stc')

In [ ]:
!pip install mne_loreta_plot

In [ ]:


# Initialize a dictionary to store the averaged Loreta results
averaged_loreta = {}

# Loop through the evoked responses
for participant, conditions in evoked_data.items():
    for condition, evoked in conditions.items():
        # Set the EEG reference to average using a projector
        evoked.set_eeg_reference(projection=True)
        
        # Apply the Loreta inverse solution
        stc = mne.minimum_norm.apply_inverse(
            evoked, inverse_operators[participant], lambda2=1.0 / 9, method='sLORETA'
        )
        
        # Add the Loreta result to the averaged_loreta dictionary
        if f'{condition}' not in averaged_loreta:
            averaged_loreta[f'{condition}'] = stc
        else:
            averaged_loreta[f'{condition}'] += stc

# Divide the summed Loreta results by the number of participants to get the average
for condition, stc in averaged_loreta.items():
    averaged_loreta[condition] = stc / len(evokeds)

# Visualize the averaged Loreta results
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Plot the averaged source-level activity for each condition
mne_loreta_plot.plot_loreta(averaged_loreta['t_avt'], 't_avt', axes=axes[0, 0])
mne_loreta_plot.plot_loreta(averaged_loreta['f_avt'], 'f_avt', axes=axes[0, 1])
mne_loreta_plot.plot_loreta(averaged_loreta['t_non'], 't_non', axes=axes[1, 0])
mne_loreta_plot.plot_loreta(averaged_loreta['f_non'], 'f_non', axes=axes[1, 1])

plt.suptitle('Averaged Loreta Analysis Results')
plt.show()

In [ ]:
# Compute the sLORETA results per condition and participant
loreta_results = {}
file_names = []
conditions = []
for file_name, evoked_dict in evoked_data.items():
    for condition, evoked in evoked_dict.items():
        # Set the EEG reference to average using a projector
        evoked.set_eeg_reference(projection=True)
        stc = mne.minimum_norm.apply_inverse(
            evoked, inverse_operators[file_name], lambda2=1.0 / 9, method='sLORETA'
        )
        loreta_results[f'{file_name}_{condition}'] = stc

        file_names.append(file_name)
        conditions.append(condition)

# Average the sLORETA results per condition
condition_stcs = {}
for condition in set(conditions):
    stcs = [
        loreta_results[f'{file_name}_{condition}']
        for file_name in set(file_names)
    ]
    condition_stcs[condition] = mne.grand_average(stcs)

In [ ]:
loreta_results

In [ ]:
# Save the file names and conditions to a file
with open(path_to_cov_loreta / 'loreta_metadata.txt', 'w') as f:
    for file_name, condition in zip(file_names, conditions):
        f.write(f'{file_name},{condition}\n')

In [ ]:
# Save the LORETA results
for file_name, stc in loreta_results.items():
    stc.save(path_to_cov_loreta / f'{file_name}_loreta.stc', overwrite = True)

In [ ]:
# Load the file names and conditions from the metadata file
file_names = []
conditions = []
with open(path_to_loreta / 'loreta_metadata.txt', 'r') as f:
    for line in f:
        file_name, condition = line.strip().split(',')
        file_names.append(file_name)
        conditions.append(condition)

In [ ]:
# Load and visualize the LORETA results
for file_name, condition in zip(file_names, conditions):
    stc = mne.read_source_estimate(f'{file_name}_{condition}_loreta.stc')
    brain = mne.viz.plot_source_estimates(stc, subject='fsaverage', surface='inflated', time_viewer=True)
    brain.show()

In [ ]:
# Load the file names, conditions, and subject from the metadata file
file_names = []
conditions = []
with open(path_to_cov_loreta / 'loreta_metadata.txt', 'r') as f:
    subject = f.readline().strip().split(',')[1]
    for line in f:
        file_name, condition = line.strip().split(',')
        file_names.append(file_name)
        conditions.append(condition)

# Create a dictionary to store the source estimates per participant and condition
participant_condition_stcs = {}

# Compute the sLORETA results per participant and condition
for file_name, condition in zip(file_names, conditions):
    stc = mne.read_source_estimate(path_to_loreta / f'{file_name}_{condition}_loreta.stc')
    participant_condition_stcs[(file_name, condition)] = stc

# Average the source estimates per condition
condition_stcs = {}
for condition in set(conditions):
    stcs = [
        participant_condition_stcs[(file_name, condition)]
        for file_name in file_names
        if (file_name, condition) in participant_condition_stcs
    ]
    condition_stcs[condition] = mne.grand_average(stcs)

# Visualize the averaged sLORETA results per condition
for condition, stc in condition_stcs.items():
    brain = mne.viz.plot_source_estimates(stc, subject=subject, surface='inflated', time_viewer=True)
    brain.show()

In [ ]:
import pyvistaqt
import ipywidgets
import mne.viz

In [ ]:
for file_name in loreta_results.keys():
    stc = mne.read_source_estimate(path_to_loreta / f'{file_name}_loreta.stc')
    # Visualize the loaded LORETA results
    brain = mne.viz.plot_source_estimates(stc, subject=subject, surface='inflated', time_viewer=True)
    brain.show()

In [ ]:
for file_name, stc in loreta_results.items():
    brain = mne.viz.plot_source_estimates(stc, subject=subject, surface='inflated', time_viewer=True)
    brain.show()